## Load Data

In [58]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
df_email = pd.read_csv('../../../data/interim/emails_cleaned.csv')

print(f"Shape: {df_email.shape}")
df_email.head()

Shape: (22866, 28)


,co_ref,time_to_renewal,crm_accreditation_completed,crm_timely_completion,crm_progress_towards_accreditation,crm_delays_in_accreditation,crm_contractor_suggested_leave,crm_contractor_engagement,crm_contractor_sentiment,crm_contractor_sentiment_score,crm_dts_or_ssip_mentioned,crm_customer_payment_intention,crm_competitors_mentioned,crm_membership_level,crm_platform_issues_raised,crm_agent_chased_contractor,crm_agent_chase_count,crm_accreditation_issues,crm_membership_overdue,crm_auto_renewal_status,crm_dissatisified_with_renewal_price,crm_customer_complained,crm_refund_mentioned,crm_negative_customer_experience,crm_dissatisfaction_with_support,crm_financial_hardship_mentioned,year,sentiment_score_missing
0,KG5766,pre_renewal,Not Discussed,Not Discussed,Not Discussed,Yes,No,Yes,neutral,50.0,No,No,No,Not Discussed,No,Yes,1.0,Not Discussed,Yes,0.0,No,No,Yes,Yes,No,Yes,2025,0
1,WO6689,pre_renewal,Not Discussed,Not Discussed,Not Discussed,No,No,Yes,satisfied,80.0,No,Yes,No,Accredited,No,No,0.0,No,No,0.0,No,No,Yes,Yes,No,Not Discussed,2026,0
2,SB4066,pre_renewal,Not Discussed,Not Discussed,Not Discussed,Yes,No,Yes,neutral,50.0,No,Yes,Not Discussed,Not Discussed,Not Discussed,Yes,3.0,Not Discussed,Yes,0.0,Not Discussed,No,Yes,Yes,No,Not Discussed,2026,0
3,YK0304,pre_renewal,Not Discussed,Not Discussed,Not Discussed,No,Not Discussed,No,Unknown,-1.0,No,Not Discussed,No,Not Discussed,Yes,No,0.0,Not Discussed,Not Discussed,0.0,Not Discussed,No,Yes,Yes,Yes,Not Discussed,2025,1
4,HH3570,pre_renewal,Not Discussed,Not Discussed,Not Discussed,No,No,Yes,satisfied,80.0,Yes,Yes,No,Accredited,No,Yes,1.0,Not Discussed,Yes,0.0,Not Discussed,No,Yes,Yes,No,Not Discussed,2025,0


## Rename year → renewal_year

This ensures the email data can be joined to billings on co_ref + renewal_year.

In [59]:
df_email['renewal_year'] = df_email['year'].astype(int)
df_email['renewal_year'].value_counts()

renewal_year
2025    15065
2026     7801
Name: count, dtype: int64

In [60]:
yes_no_cols = [
    'crm_accreditation_completed',
    'crm_timely_completion',
    'crm_progress_towards_accreditation',
    'crm_delays_in_accreditation',
    'crm_contractor_suggested_leave',
    'crm_contractor_engagement',
    'crm_dts_or_ssip_mentioned',
    'crm_customer_payment_intention',
    'crm_competitors_mentioned',
    'crm_platform_issues_raised',
    'crm_agent_chased_contractor',
    'crm_accreditation_issues',
    'crm_membership_overdue',
    'crm_dissatisified_with_renewal_price',
    'crm_customer_complained',
    'crm_refund_mentioned',
    'crm_negative_customer_experience',
    'crm_dissatisfaction_with_support',
    'crm_financial_hardship_mentioned',
]

for col in yes_no_cols:
    df_email[col] = df_email[col].map({
        'Yes'         : 1,
        'No'          : 0,
        'Not Discussed': 0,
        'Unknown'     : 0,
    })

## Encode Sentiment Column

In [61]:
sentiment_map = {
    'satisfied'    :  1,
    'neutral'      :  0,
    'dissatisfied' : -1,
    'Unknown'      :  0,
}

df_email['crm_sentiment_encoded'] = df_email['crm_contractor_sentiment'].map(sentiment_map)
df_email['crm_dissatisfied_flag'] = (df_email['crm_contractor_sentiment'] == 'dissatisfied').astype(int)
df_email['crm_satisfied_flag']    = (df_email['crm_contractor_sentiment'] == 'satisfied').astype(int)

## Encode Membership Level

In [62]:
membership_map = {
    'Not Accredited': 0,
    'Members only'  : 1,
    'In progress'   : 2,
    'Accredited'    : 3,
    'Standard'      : 1,
    'Premier'       : 3,
    'Not Discussed' : 0,
    'Unknown'       : 0,
}

df_email['crm_membership_level_ordinal'] = df_email['crm_membership_level'].map(membership_map)

## Feature Engineering

In [63]:
# High risk email — strong churn signals in one email
df_email['high_risk_email'] = (
    (df_email['crm_contractor_suggested_leave']       == 1) |
    (df_email['crm_competitors_mentioned']            == 1) |
    (df_email['crm_financial_hardship_mentioned']     == 1) |
    (df_email['crm_dissatisified_with_renewal_price'] == 1) |
    (df_email['crm_customer_complained']              == 1)
).astype(int)

# Dissatisfaction issue count per email
dissatisfaction_cols = [
    'crm_dissatisfaction_with_support',
    'crm_negative_customer_experience',
    'crm_platform_issues_raised',
    'crm_delays_in_accreditation',
    'crm_membership_overdue',
]
df_email['emails_dissatisfaction_issue_count'] = df_email[dissatisfaction_cols].sum(axis=1)

# Engagement score — positive signals
df_email['engagement_score'] = (
    df_email['crm_accreditation_completed'] +
    df_email['crm_timely_completion'] +
    df_email['crm_progress_towards_accreditation'] +
    df_email['crm_contractor_engagement'] +
    df_email['crm_customer_payment_intention']
)

# Sentiment score valid flag
df_email['sentiment_score_valid'] = (df_email['sentiment_score_missing'] == 0).astype(int)

## Aggregate per co_ref

In [64]:
agg_dict = {
    # Sentiment
    'crm_sentiment_encoded'                      : 'mean',
    'crm_dissatisfied_flag'                      : 'max',
    'crm_satisfied_flag'                         : 'max',
    'crm_contractor_sentiment_score'             : 'mean',

    # Issue flags — max = happened at least once
    'crm_delays_in_accreditation'                : 'max',
    'crm_contractor_suggested_leave'             : 'max',
    'crm_competitors_mentioned'                  : 'max',
    'crm_platform_issues_raised'                 : 'max',
    'crm_membership_overdue'                     : 'max',
    'crm_dissatisified_with_renewal_price'       : 'max',
    'crm_customer_complained'                    : 'max',
    'crm_refund_mentioned'                       : 'max',
    'crm_negative_customer_experience'           : 'max',
    'crm_dissatisfaction_with_support'           : 'max',
    'crm_financial_hardship_mentioned'           : 'max',
    'crm_dts_or_ssip_mentioned'                  : 'max',

    # Positive signals
    'crm_accreditation_completed'                : 'max',
    'crm_timely_completion'                      : 'max',
    'crm_progress_towards_accreditation'         : 'max',
    'crm_contractor_engagement'                  : 'max',
    'crm_customer_payment_intention'             : 'max',
    'crm_agent_chased_contractor'                : 'max',
    'crm_accreditation_issues'                   : 'max',

    # Numeric
    'crm_agent_chase_count'                      : 'sum',
    'crm_auto_renewal_status'                    : 'mean',
    'crm_membership_level_ordinal'               : 'max',

    # Engineered features
    'high_risk_email': 'max',
    'emails_dissatisfaction_issue_count' : 'sum',
    'engagement_score' : 'sum',
    'sentiment_score_valid' : 'mean',
}

# group by co_ref + renewal_year (not just co_ref)
df_email_agg = df_email.groupby(['co_ref', 'renewal_year']).agg(agg_dict).reset_index()

# Add email count
email_counts = df_email.groupby(['co_ref', 'renewal_year']).size().reset_index(name='email_count')
df_email_agg = df_email_agg.merge(email_counts, on=['co_ref', 'renewal_year'], how='left')

print(f"Shape after aggregation: {df_email_agg.shape}")
df_email_agg.head()

Shape after aggregation: (21362, 33)


,co_ref,renewal_year,crm_sentiment_encoded,crm_dissatisfied_flag,crm_satisfied_flag,crm_contractor_sentiment_score,crm_delays_in_accreditation,crm_contractor_suggested_leave,crm_competitors_mentioned,crm_platform_issues_raised,crm_membership_overdue,crm_dissatisified_with_renewal_price,crm_customer_complained,crm_refund_mentioned,crm_negative_customer_experience,crm_dissatisfaction_with_support,crm_financial_hardship_mentioned,crm_dts_or_ssip_mentioned,crm_accreditation_completed,crm_timely_completion,crm_progress_towards_accreditation,crm_contractor_engagement,crm_customer_payment_intention,crm_agent_chased_contractor,crm_accreditation_issues,crm_agent_chase_count,crm_auto_renewal_status,crm_membership_level_ordinal,high_risk_email,emails_dissatisfaction_issue_count,engagement_score,sentiment_score_valid,email_count
0,AA0784,2026,0.0,0,0,-1.0,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,1,1,1.0,0.0,0,0,3,0,0.0,1
1,AA0794,2025,0.0,0,0,-1.0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,2.0,0.0,3,0,1,0,0.0,1
2,AA0882,2025,0.0,0,0,50.0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,1,0,1.0,0.0,0,0,1,2,1.0,1
3,AA0923,2026,0.0,0,0,-1.0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,1.0,0.0,0,0,1,0,0.0,1
4,AA0994,2025,0.0,0,0,50.0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0.0,2.0,3,1,0,2,1.0,1


## Drop Redundant Columns

In [65]:
drop_cols = [
    'time_to_renewal',        # only pre_renewal in this file
    'sentiment_score_missing', # replaced by sentiment_score_valid
]

df_email_agg.drop(columns=[c for c in drop_cols if c in df_email_agg.columns], inplace=True)

## Final Check

In [66]:
print("Nulls: ", df_email_agg.isnull().sum()[df_email_agg.isnull().sum() > 0])

print(df_email_agg.shape)
df_email_agg.head()

Nulls:  Series([], dtype: int64)
(21362, 33)


,co_ref,renewal_year,crm_sentiment_encoded,crm_dissatisfied_flag,crm_satisfied_flag,crm_contractor_sentiment_score,crm_delays_in_accreditation,crm_contractor_suggested_leave,crm_competitors_mentioned,crm_platform_issues_raised,crm_membership_overdue,crm_dissatisified_with_renewal_price,crm_customer_complained,crm_refund_mentioned,crm_negative_customer_experience,crm_dissatisfaction_with_support,crm_financial_hardship_mentioned,crm_dts_or_ssip_mentioned,crm_accreditation_completed,crm_timely_completion,crm_progress_towards_accreditation,crm_contractor_engagement,crm_customer_payment_intention,crm_agent_chased_contractor,crm_accreditation_issues,crm_agent_chase_count,crm_auto_renewal_status,crm_membership_level_ordinal,high_risk_email,emails_dissatisfaction_issue_count,engagement_score,sentiment_score_valid,email_count
0,AA0784,2026,0.0,0,0,-1.0,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,1,1,1.0,0.0,0,0,3,0,0.0,1
1,AA0794,2025,0.0,0,0,-1.0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,2.0,0.0,3,0,1,0,0.0,1
2,AA0882,2025,0.0,0,0,50.0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,1,0,1.0,0.0,0,0,1,2,1.0,1
3,AA0923,2026,0.0,0,0,-1.0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,1.0,0.0,0,0,1,0,0.0,1
4,AA0994,2025,0.0,0,0,50.0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0.0,2.0,3,1,0,2,1.0,1


## Save

In [67]:
df_email_agg.to_csv('../../../data/interim/email_features.csv', index=False)